# Banglish → Bangla — mBART-50 fine-tuning (Kaggle)

Clean training notebook for Bongolipi. It fine-tunes `facebook/mbart-large-50`
to translate Banglish (Bangla written in Latin script) into Bangla, then
**pushes the model to the Hugging Face Hub** so it can be hosted for the app.

It can optionally merge **approved user contributions** from MongoDB into the
training data, so the model keeps improving from real usage.

## Before you run
1. **Turn on the GPU:** Kaggle → Settings (right panel) → *Accelerator* → **GPU T4 x1**.
2. **Add a Hugging Face token** (a *write* token from https://huggingface.co/settings/tokens):
   Kaggle → *Add-ons* → *Secrets* → add secret named **`HF_TOKEN`**.
3. **(Optional) Add your MongoDB URI** as a secret named **`MONGODB_URI`** to fold in
   approved contributions. Skip it to train on the base dataset only.
4. Set **`REPO_ID`** below to `your-hf-username/banglish-bangla-mbart`.
5. Run all cells.

In [ ]:
# Pinned, mutually-compatible versions — avoids the Kaggle transformers/huggingface_hub clash.
# IMPORTANT: after this cell finishes, RESTART the kernel (the circular-arrow "Restart" button,
# or Run -> Restart & clear cell outputs), then run all cells again from the top.
# This drops the older transformers/huggingface_hub that Kaggle preloads.
!pip install -q "transformers==4.46.3" "huggingface_hub==0.26.5" "datasets==3.2.0" "accelerate==1.2.1" sentencepiece pymongo

In [ ]:
import os

# ---- Configuration -------------------------------------------------------
MODEL_NAME = "facebook/mbart-large-50"
REPO_ID = "your-username/banglish-bangla-mbart"  # <-- CHANGE to your HF repo

BASE_DATASET = "SKNahin/bengali-transliteration-data"
SRC_COL = "rm"   # Banglish (romanized) column in the base dataset
TGT_COL = "bn"   # Bangla column in the base dataset

SRC_LANG = "en_XX"   # mBART code for the Latin-script source
TGT_LANG = "bn_IN"   # mBART code for Bangla

MAX_LEN = 128
EPOCHS = 3
TRAIN_BATCH = 8
EVAL_BATCH = 8
GRAD_ACCUM = 2
LR = 3e-5
MAX_SAMPLES = None   # set an int (e.g. 50000) for a quick first run; None = all

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# ---- Secrets (Kaggle Secrets, with env fallback) -------------------------
HF_TOKEN = ""
MONGODB_URI = ""
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    try:
        HF_TOKEN = _us.get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        MONGODB_URI = _us.get_secret("MONGODB_URI")
    except Exception:
        pass
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    MONGODB_URI = os.environ.get("MONGODB_URI", "")

assert HF_TOKEN, "Set an HF_TOKEN secret (a write token) to push the model."
print("HF token loaded.", "MongoDB URI set." if MONGODB_URI else "No MongoDB URI (base dataset only).")

## 1. Load the data
The base transliteration dataset, optionally merged with approved contributions.

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets

raw = load_dataset(BASE_DATASET)
print("Splits:", raw)
base = raw["train"]
print("Columns:", base.column_names)
assert SRC_COL in base.column_names and TGT_COL in base.column_names, (
    f"Expected columns '{SRC_COL}' and '{TGT_COL}'. Update SRC_COL/TGT_COL to match the printed columns."
)
base = base.select_columns([SRC_COL, TGT_COL])
if MAX_SAMPLES:
    base = base.select(range(min(MAX_SAMPLES, len(base))))
print("Base pairs:", len(base))

In [ ]:
# ---- Optional: merge approved contributions from MongoDB -----------------
contrib = None
if MONGODB_URI:
    from pymongo import MongoClient
    client = MongoClient(MONGODB_URI)
    db = client.get_database("bongolipi")
    rows = list(
        db.contributions.find(
            {"isApproved": True},
            {"banglish_text": 1, "bangla_text": 1, "_id": 0},
        )
    )
    pairs = [
        (r.get("banglish_text", "").strip(), r.get("bangla_text", "").strip())
        for r in rows
    ]
    pairs = [(s, t) for s, t in pairs if s and t]
    if pairs:
        contrib = Dataset.from_dict(
            {SRC_COL: [s for s, _ in pairs], TGT_COL: [t for _, t in pairs]}
        )
        print("Approved contribution pairs:", len(contrib))
    else:
        print("No approved contributions found.")

data = concatenate_datasets([base, contrib]) if contrib else base
print("Total pairs:", len(data))

## 2. Tokenizer & model

In [ ]:
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)

In [ ]:
def preprocess(examples):
    model_inputs = tokenizer(
        examples[SRC_COL], max_length=MAX_LEN, truncation=True
    )
    labels = tokenizer(
        text_target=examples[TGT_COL], max_length=MAX_LEN, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

split = data.train_test_split(test_size=0.05, seed=42)
tokenized = split.map(
    preprocess, batched=True, remove_columns=[SRC_COL, TGT_COL]
)
print(tokenized)

## 3. Train

In [ ]:
import torch
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

## 4. Sanity-check a translation
mBART must be told to start decoding in Bangla via `forced_bos_token_id`.

In [ ]:
def translate(text):
    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG],
        max_length=MAX_LEN,
        num_beams=5,
    )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

for s in ["ajke amar mon onek bhalo", "tumi kemon acho", "ami bhat khai"]:
    print(s, "->", translate(s))

## 5. Push to the Hugging Face Hub
This publishes the model + tokenizer so the app can host it.

In [ ]:
model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{REPO_ID}")

## Done
The model is now on the Hub at **`REPO_ID`**. Send me that repo id and I'll
build the inference Space (`POST /banglish` → `{ generated_text }`) and wire the
app to use it (with Groq as the fallback).

> Note for hosting/inference: always generate with
> `forced_bos_token_id = tokenizer.lang_code_to_id['bn_IN']` and set
> `tokenizer.src_lang = 'en_XX'`, exactly like the `translate()` cell above.